<a href="https://colab.research.google.com/github/cbonnin88/LimeLight/blob/main/Data_generation_and_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 56.3 MB/s eta 0:00:00


In [4]:
import polars as pl
import numpy as np
from faker import Faker
import random

In [6]:
locales = {
    'France':'fr_FR',
    'UK':'en_GB',
    'Spain':'es_ES',
    'Denmark':'da_DK',
    'Germany':'de_DE',
    'Austria':'de_AT',
    'Belgium':'fr_BE',
    'The Netherlands':'nl_NL'
}

faker_country = {country: Faker(locale) for country, locale in locales.items()}

In [7]:
n = 12500

# **Office Dimension Table**

In [9]:
offices_data = {
    'office_id': [1,2,3,4,5,6,7,8],
    'city': ["Paris", "London", "Madrid", "Copenhagen", "Berlin", "Vienna", "Brussels",'Amsterdam'],
    'country': ["France", "UK", "Spain", "Denmark", "Germany", "Austria", "Belgium",'The Netherlands']
}
offices = pl.DataFrame(offices_data)
id_to_country = dict(zip(offices_data['office_id'], offices_data['country']))

# **Job Title Pools**

In [10]:
job_map = {
    "Engineering": ["Data Engineer", "Analytics Engineer", "Frontend Engineer", "Backend Engineer"],
    "Operations": ["Solar Installer", "Maintenance Tech", "Operations Manager", "Site Supervisor"],
    "ESG Strategy": ["Climate Risk & Transition Manager", "Structural Engineer", "Grid Specialist", "Project Engineer"],
    "Sales": ["Account Manager", "Sales Representative", "Business Development", "Sales Director"],
    "Human Resources": ["People Operations Officer", "Talent Acquisition Partner", "Compensation Analyst", "Human Resources Business Partner",'People Analyst'],
    "Product":["Product Manager","Product Analyst","Product Engineer","Product Owner","Product Designer"],
    'Finance': ['Financial Analyst','Business Analyst','Legal Counsel','Accountant'],
    'Leadership': ['Cheif Executive Officer','Chief Financial Officer','Chief People Officer','Chief Technical Officer','Head of Marketing','Head of Research']
    }

# **Generate Core Data**

In [16]:
office_assignments = np.random.choice(offices_data['office_id'],n)
all_titles = list(job_map.keys())
genders = np.random.choice(["Female", "Male", "Non-Binary"], n, p=[0.485, 0.485, 0.03])

# **Generate localized names based on office assignment**

In [18]:
first_names = []
last_names = []

for i in range(n):
    country = id_to_country[office_assignments[i]]
    gender = genders[i]
    fake = faker_country[country]

    if gender == "Male":
        first_names.append(fake.first_name_male())
    elif gender == "Female":
        first_names.append(fake.first_name_female())
    else:
        # Non-Binary: random choice of any first name
        first_names.append(fake.first_name())

    last_names.append(fake.last_name())

# **Building the Employees DataFrame**

In [19]:
employees_df = pl.DataFrame({
    'emp_id': range(10001,10001 + n),
    'first_name': first_names,
    'last_name': last_names,
    'gender': genders,
    'office_id': office_assignments,
    'job_title': np.random.choice(all_titles,n),
    'tenure': np.random.randint(1,7,n),
    'remote_type': np.random.choice(['remote','office','hybrid'],n,p=[0.60,0.10,0.30]),
    'performance': np.random.choice(['Top','average','bottom'], n, p=[0.2,0.6,0.2]),
    'hours': np.random.randint(20,38,n),

})

# **Adding a Department, contract_type and manager logic**

In [22]:
employees_df = employees_df.with_columns([
    # Creating the 'dept' column based on the job title
    pl.col('job_title').alias('department'),
    pl.when(pl.col('hours') >=32).then(pl.lit('Full-time')).otherwise(pl.lit('Part-time')).alias('contract_type')
]).with_columns([
    # Everyone in Leadership is a manager; 10% of others are managers
    pl.when((pl.col('department')=='Leadership') | (np.random.random(n) > 0.90))
    .then(pl.lit('yes')).otherwise(pl.lit('no')).alias('manager')
])

# **Create Compensation DataFrame**

In [27]:
compensation_df = employees_df.select(["emp_id", "department", "hours"]).with_columns([
    pl.when(pl.col("department") == "Leadership").then(pl.lit(np.random.uniform(120000, 190000)))
    .when(pl.col("department") == "Engineering").then(pl.lit(np.random.uniform(75000, 115000)))
    .otherwise(pl.lit(np.random.uniform(45000, 90000)))
    .alias("base_salary")
]).with_columns([
    ((pl.col("base_salary") / pl.col("hours")) * 37).round(2).alias("FTE_salary")
]).with_columns([
    pl.when(pl.col("department") == "Leadership").then(pl.lit("L1"))
    .when(pl.col("FTE_salary") > 100000).then(pl.lit("L2"))
    .when(pl.col("FTE_salary") > 80000).then(pl.lit("L3"))
    .when(pl.col("FTE_salary") > 60000).then(pl.lit("L4"))
    .otherwise(pl.lit("L5"))
    .alias("salary_band")
])

In [25]:
display(employees_df.head())

emp_id,first_name,last_name,gender,office_id,job_title,tenure,remote_type,performance,hours,department,contract_type,manager
i64,str,str,str,i64,str,i64,str,str,i64,str,str,str
10001,"""Henriette""","""Langlois""","""Female""",1,"""Sales""",5,"""hybrid""","""bottom""",33,"""Sales""","""Full-time""","""no"""
10002,"""Gabrielle""","""Mace""","""Female""",1,"""ESG Strategy""",3,"""remote""","""average""",34,"""ESG Strategy""","""Full-time""","""no"""
10003,"""Sofie""","""Bartels""","""Female""",8,"""Product""",3,"""remote""","""average""",36,"""Product""","""Full-time""","""no"""
10004,"""Bertrand""","""Morin""","""Male""",1,"""ESG Strategy""",5,"""remote""","""average""",37,"""ESG Strategy""","""Full-time""","""no"""
10005,"""Piotr""","""Schleich""","""Male""",5,"""Finance""",3,"""remote""","""average""",20,"""Finance""","""Part-time""","""no"""


In [28]:
display(compensation_df.head())

emp_id,department,hours,base_salary,FTE_salary,salary_band
i64,str,i64,f64,f64,str
10001,"""Sales""",33,49549.630841,55555.65,"""L5"""
10002,"""ESG Strategy""",34,49549.630841,53921.66,"""L5"""
10003,"""Product""",36,49549.630841,50926.01,"""L5"""
10004,"""ESG Strategy""",37,49549.630841,49549.63,"""L5"""
10005,"""Finance""",20,49549.630841,91666.82,"""L3"""


In [30]:
display(offices)

office_id,city,country
i64,str,str
1,"""Paris""","""France"""
2,"""London""","""UK"""
3,"""Madrid""","""Spain"""
4,"""Copenhagen""","""Denmark"""
5,"""Berlin""","""Germany"""
6,"""Vienna""","""Austria"""
7,"""Brussels""","""Belgium"""
8,"""Amsterdam""","""The Netherlands"""


# **Ready for Google Sheets (Adhoc People Analytics)**

In [32]:
employees_df.write_csv('employees.csv')
compensation_df.write_csv('compensation.csv')
offices.write_csv('offices.csv')

print('All Files exported')

All Files exported
